# PhysAR-IDS -- NB6c (v6): burst-clustered intervals for the autoencoder gate and the layered pipeline

**Why.** The paper gives burst-clustered 95% intervals for all four attackers and both adversarially trained models, but not for the gate. NB6 and NB6b saved only per-seed means (no row-level verdicts), so intervals cannot be computed from their files. This notebook **reloads** the five saved gates (`ae_gate_v6_seed{s}.pt`) and the NB5b white-box-trained classifiers (`xgb_robust_v6_seed{s}.json`), repeats the white-box attacks against them **on the same rows as NB6 / NB6b**, keeps the row-level verdicts, and resamples bursts.

**Attacks (cyber features only, 1.0 and 0.3 sigma: both columns of Table VII in one run).**
* *Classifier alone / gate-unaware:* the tree-greedy white-box search against Layer 1 only (as NB6, 60 rounds). The row is then scored by Layer 1 alone and by the pipeline (Layer 1 and gate).
* *Fully adaptive:* the same search on the joint score `min(margin gap, (tau - error) / tau)`, for up to **300 rounds** (NB6b's long run). Rows are NB6b's 500 per class (a fixed random subset of NB6's).
* *Clean rows:* the gate and Layer 1 on the unperturbed test set.

**Reported, each with a 95% interval from resampling bursts (class, PLC, timestamp), 2,000 draws, pooled over the five seeds.**
1. The gate alone: detection of attack rows (resampling attack bursts), false alarms on benign rows (resampling benign bursts), detection of FDI; Layer 1 alone and the pipeline on clean test data.
2. Under the white-box attacks: Layer-1-only evasion, gate-unaware pipeline evasion, fully adaptive pipeline evasion, per class, and the differences (adaptive minus classifier alone; adaptive minus gate-unaware; classifier alone minus gate-unaware), with paired seed tests (Holm).
3. **Independence ratio** = share of rows that pass both layers divided by (share passing Layer 1) x (share passing the gate), on the attacked rows. 1 = independent layers; above 1 = the survivors are more numerous than independence predicts.

**Needs** `ae_gate_v6_seed{s}.pt`, `xgb_robust_v6_seed{s}.json` and the data files in `SmartGrid_Dissertation`. **Runtime:** about 1 to 2 hours per seed and budget on the GPU (the 300-round search dominates). Each budget is saved as soon as it finishes, and the run resumes. A file saved by an earlier version (1.0 sigma only) is accepted: only the missing 0.3 sigma column is run.

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def find_project_dir(env_names=('PHYSAR_DIR',), marker='phase1_data_v6.pkl'):
    """Locate the folder that holds the pipeline files, wherever the notebook is run from.
    Order: environment variable(s) -> the current folder and its parents (and a SmartGrid_Dissertation folder next to or inside each) -> the usual places under the home folder."""
    from pathlib import Path
    cands = []
    for n in env_names:
        if os.environ.get(n): cands.append(Path(os.environ[n]).expanduser())
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents][:6]:
        cands += [p, p / 'SmartGrid_Dissertation'] + [p / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    home = Path.home()
    for extra in ('/content', '/content/drive/MyDrive', '/content/drive/MyDrive/smartgrid1', '/content/drive/MyDrive/smartgrid1/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1', '/content/drive/MyDrive/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1/SmartGrid_Dissertation', '/content/drive/My Drive', '/content/drive/My Drive/smartgrid1', '/content/drive/My Drive/smartgrid1/SmartGrid_Dissertation', '/content/drive/My Drive/smartrid1/SmartGrid_Dissertation', '/kaggle/working'):
        cands += [Path(extra), Path(extra) / 'SmartGrid_Dissertation']
    for base in ('Desktop', 'Documents', 'Downloads', 'OneDrive', 'OneDrive/Desktop', 'OneDrive/Documents', ''):
        b = home / base if base else home
        cands += [b / 'SmartGrid_Dissertation'] + [b / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    seen = set()
    for c in cands:
        try: c = c.resolve()
        except OSError: continue
        if c in seen: continue
        seen.add(c)
        if (c / marker).is_file(): return str(c)
    for root in [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')] + [home / b for b in ('Desktop', 'Documents', 'Downloads', 'OneDrive')]:   # shallow search, depth <= 3
        if not root.is_dir(): continue
        for dp, dn, fn in os.walk(root):
            if marker in fn: return dp
            if Path(dp).relative_to(root).parts.__len__() >= 3: dn[:] = []
    raise FileNotFoundError(f"Could not find {marker}. Set the environment variable PHYSAR_DIR to the folder that holds the pipeline files "
                            f"(e.g. in a cell, before running: os.environ['PHYSAR_DIR'] = r'C:\\path\\to\\SmartGrid_Dissertation'), or upload the files to the same folder as the notebook (on Colab: /content, or mount Google Drive first).")
PROJECT_DIR = find_project_dir(('NB6C_DIR', 'PHYSAR_DIR')); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = bool(int(os.environ.get('NB6C_SMOKE', '0')))

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)


# ---- NB6: Attack 3 -- adaptive attacker against the WHOLE pipeline (Layer 1 classifier + Layer 2 autoencoder gate) ----
LAYER1 = {'rob': 'xgb_robust_{v}_seed{s}.json', 'def': 'xgb_defended_{v}_seed{s}.json'}      # NB5b robust model (primary) and NB5 defended model
AW_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '1.0': 1.0}      # sigma, cyber features only (same attacker-controlled set as NB3c / NB5)
N_AW_PER_CLASS = 1000                                    # rows per attack class (a random subset of NB5's 2,000 per class)
MAX_ITERS_AWARE = 100                                    # rounds for the greedy search against the pipeline
GATE_SCALE = 1.0                                         # weight of the gate term in the joint score min(margin gap, GATE_SCALE * (tau - err) / tau)
N_FOLDS = 5; GATE_EPOCHS = 30; GATE_PCTL = 95; GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_CYBER_BOOST = 1.0
RUN_PPO_AWARE = True; PPO_AWARE_MODELS = ['rob']; PPO_AWARE_BUDGETS = {'0.3': 0.3, '1.0': 1.0}      # black-box PPO retrained against the pipeline's hard-label verdict
PPO_UNAWARE_TAGS = ['0.3', '1.0', 'unconstrained']      # the saved NB4 agents (trained against the standard classifier, gate-unaware)

# ---- NB6c ----
RUN_BUDGETS = os.environ.get('NB6C_BUDGETS', '1.0,0.3').split(',')      # both columns of Table VII in one run; a saved seed that lacks a budget runs only that budget
LONG_ITERS = 300                       # rounds for the fully adaptive search (NB6b)
N_LONG_PER_CLASS = 500                 # NB6b's rows: a fixed random subset of NB6's 1,000 per class
N_BOOT = 2000
if SMOKE_TEST: SEEDS = SEEDS[:1]; K_CAND = 4; MAX_ITERS = 6; LONG_ITERS = 8; N_WB_PER_CLASS = 60; N_AW_PER_CLASS = 40; N_LONG_PER_CLASS = 25; N_BOOT = 200; GATE_EPOCHS = 2

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
print('Working directory:', os.getcwd())

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

In [ ]:
class AE(nn.Module):
    def __init__(self, n_features, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_features))
    def forward(self, x): return self.decoder(self.encoder(x))
FW = torch.ones(N_FEAT, device=DEVICE); FW[torch.as_tensor(CYBER_IDX, device=DEVICE)] = GATE_CYBER_BOOST
def wmse(recon, target): return (((recon - target) ** 2) * FW).sum(1) / FW.sum()

class Gate:
    def __init__(self, models, tau): self.models = models; self.tau = float(tau)
    @torch.no_grad()
    def score(self, X, bs=65536):
        X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
        for s in range(0, len(X), bs):
            b = torch.from_numpy(X[s:s + bs]).to(DEVICE); out[s:s + bs] = torch.stack([wmse(m(b), b) for m in self.models]).mean(0).double().cpu().numpy()
        return out
    def flag(self, X): return self.score(X) > self.tau

def train_gate(seed, X_benign):
    from sklearn.model_selection import KFold
    oof = np.zeros(len(X_benign)); models = []
    for k, (tr, ho) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(X_benign)):
        torch.manual_seed(seed * 10 + k); m = AE(N_FEAT).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        ds = TensorDataset(torch.from_numpy(X_benign[tr]).float()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed * 10 + k))
        for ep in range(GATE_EPOCHS):
            m.train()
            for (xb,) in loader: xb = xb.to(DEVICE); opt.zero_grad(); wmse(m(xb), xb).mean().backward(); opt.step()
        m.eval(); models.append(m); oof[ho] = Gate([m], 0).score(X_benign[ho])
    return Gate(models, np.percentile(oof, GATE_PCTL)), oof

class PipelineOracle:
    """hard-label verdict of the whole pipeline: a row Layer 1 calls benign but the gate flags is reported as an attack (class 1)."""
    def __init__(self, clf, gate): self.clf = clf; self.gate = gate; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X); p = self.clf.predict(X)
        return np.where((p == 0) & self.gate.flag(X), 1, p)

class PipelineMargin:
    """the quantity the fully adaptive white-box attacker maximises: min(classifier margin gap, GATE_SCALE * (tau - error) / tau); > 0 <=> evades both layers.
    Returned as a 2-column array so that the existing gap_of(z) = z[:,0] - max(z[:,1:]) equals this score."""
    def __init__(self, clf_gpu, gate): self.m = Margin(clf_gpu); self.gate = gate
    def __call__(self, X):
        s = np.minimum(gap_of(self.m(X)), GATE_SCALE * (self.gate.tau - self.gate.score(X)) / self.gate.tau)
        return np.stack([s, np.zeros_like(s)], 1)
print('gate, pipeline oracle and joint-score margin defined')

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]                           # NB5 / NB5b / NB5c rows
rng1 = np.random.RandomState(1); sub = np.sort(np.concatenate([rng1.choice(np.flatnonzero(y_wb == c), N_AW_PER_CLASS, replace=False) for c in range(1, 6)]))   # random subset (not the earliest rows)
X_aw, y_aw = X_wb[sub], y_wb[sub]
def pcs(ev, y): return np.array([ev[y == c].mean() for c in range(1, 6)])
def load_model(path, seed):
    gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu.load_model(path)
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
rng2 = np.random.RandomState(2); sub2 = np.sort(np.concatenate([rng2.choice(np.flatnonzero(y_aw == c), N_LONG_PER_CLASS, replace=False) for c in range(1, 6)])); X_lg, y_lg = X_aw[sub2], y_aw[sub2]
burst_lg = test_burst[pick][sub][sub2]
print(f'rows used here: {len(X_lg)} ({N_LONG_PER_CLASS} per class) in {len(np.unique(burst_lg))} of {len(np.unique(test_burst))} attack bursts | budgets {RUN_BUDGETS} | up to {LONG_ITERS} rounds')
print(f'attack rows used by the white-box attackers: {len(X_aw)} ({N_AW_PER_CLASS} per class) | budgets {AW_BUDGETS} | PGD / PPO use all {len(X_atk_test):,} test attack rows')

## Gate reload, verdicts, and the per-seed run (resumable)

In [ ]:
def verdict(clf_cpu, gate, X):
    ev1 = clf_cpu.predict(X) == 0; fl = gate.flag(X); return ev1, fl, ev1 & ~fl
def load_gate(seed):
    gp = f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt'
    if SMOKE_TEST and not os.path.exists(gp):                      # smoke test only: make a tiny gate so the pipeline can be exercised
        g_, _ = train_gate(seed, X_train[y_train == 0]); torch.save({'state': [m.state_dict() for m in g_.models], 'tau': g_.tau, 'cfg': {}}, gp)
    ck = torch.load(gp, map_location=DEVICE); ms_ = []
    for st in ck['state']: m = AE(N_FEAT).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
    return Gate(ms_, ck['tau'])
def get_model(seed):
    mp = f'xgb_robust_{PIPELINE_VERSION}_seed{seed}.json'
    if SMOKE_TEST and not os.path.exists(mp):
        c_ = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); c_.fit(X_train, y_train, verbose=False); c_.save_model(mp)
    return load_model(mp, seed)
R = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack3c_seed{seed}_{PIPELINE_VERSION}.pkl'
    def mk_fp(rb): return repr((rb, LONG_ITERS, MAX_ITERS, N_LONG_PER_CLASS, N_AW_PER_CLASS, N_WB_PER_CLASS, GATE_SCALE, K_CAND, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb6c-v1'))
    cfg_fp = mk_fp(RUN_BUDGETS); fp_base = mk_fp(None).replace('None', 'BUDGETS'); r = {}
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') in (cfg_fp, mk_fp(['1.0']), mk_fp(['0.3', '1.0']), mk_fp(['1.0', '0.3'])) or saved.get('fp_base') == fp_base: r = saved['R']      # also accepts files written with a different budget list
    todo_b = [b_ for b_ in RUN_BUDGETS if f'aware_{b_}' not in r]
    if not todo_b: R[seed] = r; print(f'seed {seed}: result file found for budgets {RUN_BUDGETS} -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}: budgets to run {todo_b} (already saved: {[b_ for b_ in RUN_BUDGETS if b_ not in todo_b]})', flush=True)
    gate = load_gate(seed); sc = gate.score(X_test); gpu_m, cpu_m = get_model(seed); TH = get_thresholds(gpu_m); mg = Margin(gpu_m); pm = PipelineMargin(gpu_m, gate)
    pred_te = cpu_m.predict(X_test)
    r['clean'] = {'tau': gate.tau, 'gate_flag': sc > gate.tau, 'l1_benign': pred_te == 0, 'pipeline_pass': (pred_te == 0) & ~(sc > gate.tau)}
    ev1c, flc, evpc = verdict(cpu_m, gate, X_lg); r['clean_rows'] = {'ev1': ev1c, 'fl': flc, 'evp': evpc}
    print(f'  gate: tau {gate.tau:.5f} | clean FPR {(sc[y_test == 0] > gate.tau).mean():.4f} | detection {(sc[atk_te] > gate.tau).mean():.4f}', flush=True)
    if seed == SEEDS[0]: selftest(gpu_m, mg, TH)
    for bt in todo_b:
        B = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}[bt]
        ts = time.time(); xa = whitebox_trace(mg, X_lg, y_lg, CYBER_IDX, TH, B, max_iters=MAX_ITERS, verbose=False)[0]; ev1, fl, evp = verdict(cpu_m, gate, xa)
        r[f'unaware_{bt}'] = {'ev1': ev1, 'fl': fl, 'evp': evp}; print(f'  [{bt} sigma] gate-unaware: Layer 1 only {ev1.mean():.3f} -> pipeline {evp.mean():.3f} (gate flags {fl[ev1].mean():.3f} of Layer-1 evasions) | {(time.time() - ts) / 60:.1f} min', flush=True)
        ts = time.time(); xa, ga, ie, st, tr = whitebox_trace(pm, X_lg, y_lg, CYBER_IDX, TH, B, max_iters=LONG_ITERS, verbose=False); ev1, fl, evp = verdict(cpu_m, gate, xa)
        r[f'aware_{bt}'] = {'ev1': ev1, 'fl': fl, 'evp': evp, 'still_improving': float((~st & ~evp).mean()), 'stalled': float((st & ~evp).mean()), 'rounds': len(tr), 'search_vs_cpu_disagree': float(((ga > 0) != evp).mean())}
        print(f'  [{bt} sigma] fully adaptive: pipeline {evp.mean():.3f} per-class {np.round([evp[y_lg == c].mean() for c in range(1, 6)], 3).tolist()} | still improving {r[f"aware_{bt}"]["still_improving"]:.3f} | rounds {len(tr)} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'fp_base': fp_base, 'R': r}, pf)
    del gpu_m, cpu_m, mg, pm; torch.cuda.empty_cache()
    R[seed] = r; joblib.dump({'fp': cfg_fp, 'fp_base': fp_base, 'R': r}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## Burst-clustered intervals

Attack rows are resampled by attack burst (350 in the test period); benign rows by benign burst. Estimates are means over the five seeds of row-level rates; intervals are the 2.5th and 97.5th percentiles of 2,000 resamples, the same procedure as for the attackers.

In [ ]:
full_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0]
ben = y_test == 0
def prep(ids):
    u = np.unique(ids); return np.searchsorted(u, ids), len(u)
bi_atk_full, nb_atk = prep(test_burst); bi_ben, nb_ben = prep(full_burst[ben])
bi_lg = np.searchsorted(np.unique(test_burst), burst_lg)
IDX = {nb_atk: np.random.RandomState(42).randint(0, nb_atk, size=(N_BOOT, nb_atk)), nb_ben: np.random.RandomState(43).randint(0, nb_ben, size=(N_BOOT, nb_ben))}
def bstat(vals, bi, nb):
    idx = IDX[nb]; cnt = np.bincount(bi, minlength=nb).astype(float); draws = []
    for v in vals: draws.append(np.bincount(bi, weights=v.astype(float), minlength=nb)[idx].sum(1) / cnt[idx].sum(1))
    d = np.mean(draws, 0); return float(np.mean([v.mean() for v in vals])), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def bratio(parts, bi, nb):
    """parts: per seed (ev1, nfl) boolean arrays; ratio = P(both) / (P(ev1) P(not flagged))."""
    idx = IDX[nb]; cnt = np.bincount(bi, minlength=nb).astype(float); n = cnt[idx].sum(1); draws = []; point = []
    for ev1, nfl in parts:
        sw = lambda v: np.bincount(bi, weights=v.astype(float), minlength=nb)[idx].sum(1)
        draws.append((sw(ev1 & nfl) / n) / ((sw(ev1) / n) * (sw(nfl) / n))); point.append((ev1 & nfl).mean() / (ev1.mean() * nfl.mean()))
    d = np.nanmean(draws, 0); return float(np.nanmean(point)), float(np.nanpercentile(d, 2.5)), float(np.nanpercentile(d, 97.5))
rows = []
def add(label, est): rows.append({'quantity': label, 'estimate': round(est[0], 4), 'lo': round(est[1], 4), 'hi': round(est[2], 4)})
atk_names = class_names[1:]
# 1. clean test data
add('Gate alone: detection of attack rows (clean)', bstat([R[s]['clean']['gate_flag'][atk_te] for s in SEEDS], bi_atk_full, nb_atk))
add('Gate alone: false alarms on benign rows', bstat([R[s]['clean']['gate_flag'][ben] for s in SEEDS], bi_ben, nb_ben))
for c in range(1, 6):
    m = y_atk_test == c; add(f'Gate alone: detection of {atk_names[c - 1]} rows (clean)', bstat([R[s]['clean']['gate_flag'][atk_te][m] for s in SEEDS], bi_atk_full[m], nb_atk))
add('Layer 1 alone: false alarms on benign rows', bstat([~R[s]['clean']['l1_benign'][ben] for s in SEEDS], bi_ben, nb_ben))
add('Pipeline: false alarms on benign rows', bstat([~R[s]['clean']['pipeline_pass'][ben] for s in SEEDS], bi_ben, nb_ben))
add('Layer 1 alone: detection of attack rows (clean)', bstat([~R[s]['clean']['l1_benign'][atk_te] for s in SEEDS], bi_atk_full, nb_atk))
add('Pipeline: detection of attack rows (clean)', bstat([~R[s]['clean']['pipeline_pass'][atk_te] for s in SEEDS], bi_atk_full, nb_atk))
# 2. white-box attacks
for bt in RUN_BUDGETS:
    U = [R[s][f'unaware_{bt}'] for s in SEEDS]; A = [R[s][f'aware_{bt}'] for s in SEEDS]; L = f'{bt} sigma'
    add(f'{L}: classifier alone, attacker targets Layer 1', bstat([u['ev1'] for u in U], bi_lg, nb_atk))
    add(f'{L}: pipeline, gate-unaware attacker', bstat([u['evp'] for u in U], bi_lg, nb_atk))
    add(f'{L}: pipeline, fully adaptive attacker', bstat([a['evp'] for a in A], bi_lg, nb_atk))
    add(f'{L}: fully adaptive minus classifier alone', bstat([a['evp'].astype(float) - u['ev1'] for a, u in zip(A, U)], bi_lg, nb_atk))
    add(f'{L}: fully adaptive minus gate-unaware pipeline', bstat([a['evp'].astype(float) - u['evp'] for a, u in zip(A, U)], bi_lg, nb_atk))
    add(f'{L}: classifier alone minus gate-unaware pipeline (what the gate removes if ignored)', bstat([u['ev1'].astype(float) - u['evp'] for u in U], bi_lg, nb_atk))
    for c in range(1, 6):
        m = y_lg == c
        add(f'{L}: fully adaptive pipeline, {atk_names[c - 1]}', bstat([a['evp'][m] for a in A], bi_lg[m], nb_atk))
        add(f'{L}: classifier alone, {atk_names[c - 1]}', bstat([u['ev1'][m] for u in U], bi_lg[m], nb_atk))
    add(f'{L}: independence ratio, gate-unaware rows', bratio([(u['ev1'], ~u['fl']) for u in U], bi_lg, nb_atk))
    add(f'{L}: independence ratio, fully adaptive rows', bratio([(a['ev1'], ~a['fl']) for a in A], bi_lg, nb_atk))
add('Clean attack rows (no attack): independence ratio', bratio([(R[s]['clean_rows']['ev1'], ~R[s]['clean_rows']['fl']) for s in SEEDS], bi_lg, nb_atk))
ci_table = pd.DataFrame(rows); pd.set_option('display.width', 320); pd.set_option('display.max_colwidth', 100); print(ci_table.to_string(index=False))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
pr = []
for bt in RUN_BUDGETS:
    for lab, f1, f2 in (('fully adaptive vs classifier alone', lambda s: R[s][f'aware_{bt}']['evp'].mean(), lambda s: R[s][f'unaware_{bt}']['ev1'].mean()),
                        ('fully adaptive vs gate-unaware pipeline', lambda s: R[s][f'aware_{bt}']['evp'].mean(), lambda s: R[s][f'unaware_{bt}']['evp'].mean()),
                        ('classifier alone vs gate-unaware pipeline', lambda s: R[s][f'unaware_{bt}']['ev1'].mean(), lambda s: R[s][f'unaware_{bt}']['evp'].mean())):
        a = np.array([f1(s) for s in SEEDS]); b = np.array([f2(s) for s in SEEDS]); d = a - b
        pr.append({'budget': f'{bt} sigma', 'comparison': lab, 'first': a.mean(), 'second': b.mean(), 'diff': d.mean(), 'p_raw': (1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else float(stats.ttest_rel(a, b).pvalue)) if len(d) > 1 else float('nan')})
paired = pd.DataFrame(pr)
if len(paired) and paired.p_raw.notna().all(): paired['p_holm'] = holm(paired.p_raw.values)
print('\nPaired t-tests over seeds (model variance only):'); print(paired.round(4).to_string(index=False))
srch = pd.DataFrame([{'budget': f'{bt} sigma', 'still improving at the round limit': np.mean([R[s][f'aware_{bt}']['still_improving'] for s in SEEDS]), 'stalled': np.mean([R[s][f'aware_{bt}']['stalled'] for s in SEEDS]), 'search vs CPU disagreement': np.mean([R[s][f'aware_{bt}']['search_vs_cpu_disagree'] for s in SEEDS])} for bt in RUN_BUDGETS]); print('\nSearch health of the fully adaptive attack:'); print(srch.round(4).to_string(index=False))

## Save

In [ ]:
tables = {'burst_ci': ci_table, 'paired': paired, 'search_health': srch}
for k_, t_ in tables.items(): t_.to_csv(f'attack3c_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'RUN_BUDGETS': RUN_BUDGETS, 'LONG_ITERS': LONG_ITERS, 'tables': tables}, f'attack3c_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB6c', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'RUN_BUDGETS': RUN_BUDGETS, 'LONG_ITERS': LONG_ITERS, 'N_BOOT': N_BOOT, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb6c_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack3c_results_{PIPELINE_VERSION}.pkl', f'attack3c_burst_ci_{PIPELINE_VERSION}.csv', f'attack3c_paired_{PIPELINE_VERSION}.csv', f'nb6c_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)